## Downloads Documents

In [12]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os
import pdfplumber
import pandas as pd

# Base URL of the website
base_url = 'https://rbi.org.in/Scripts/AnnualPublications.aspx?head=State+Finances+%3a+A+Study+of+Budgets'

# Set headers to mimic a browser request
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Define the path where you want to store the files
download_folder = r'C:/Users/chandresh.kanyal/Downloads'  # Change this to your desired folder path

# Ensure the directory exists
if not os.path.exists(download_folder):
    os.makedirs(download_folder)

# Send a GET request to the website
response = requests.get(base_url, headers=headers)

# Check if the request was successful
if response.status_code == 200:
    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.content, 'html.parser')

    # Find the table that contains the row for Statement 21
    table = soup.find('table')  # You may need to adjust this if there are multiple tables

    # Iterate through each row in the table to find the relevant one
    for row in table.find_all('tr'):
        cells = row.find_all('td')
        if len(cells) > 0 and 'Statement 21: Market Borrowings of State Governments' in cells[0].text:
            # This is the correct row
            print("Found the row for Statement 21")

            # Look for the download links (Excel and PDF)
            excel_link = None
            pdf_link = None

            # Iterate through the cells to find the download links (Excel and PDF)
            for cell in cells:
                # Find all anchor tags with the 'href' attribute (links)
                links = cell.find_all('a', href=True)
                for link in links:
                    href = link['href']
                    print(f"Found link: {href}")  # Debugging: print each link found
                    
                    # Check if the link is an Excel or PDF file
                    if '.xlsx' in href:
                        excel_link = urljoin(base_url, href)  # Resolve relative URL
                    elif '.pdf' in href:
                        pdf_link = urljoin(base_url, href)  # Resolve relative URL

            # Check and download the Excel file if the link is found
            if excel_link:
                print(f"Found Excel file link: {excel_link}")
                try:
                    excel_response = requests.get(excel_link, headers=headers, allow_redirects=True)
                    print(f"Excel response status: {excel_response.status_code}")  # Debugging response status
                    excel_response.raise_for_status()  # Raise an error for bad status codes
                    excel_file_path = os.path.join(download_folder, 'Statement_21_Market_Borrowings.xlsx')
                    with open(excel_file_path, 'wb') as f:
                        f.write(excel_response.content)
                    print(f"Excel file downloaded successfully to: {excel_file_path}")
                except requests.exceptions.RequestException as e:
                    print(f"Error downloading Excel file: {e}")

            # Check and download the PDF file if the link is found
            if pdf_link:
                print(f"Found PDF file link: {pdf_link}")
                try:
                    pdf_response = requests.get(pdf_link, headers=headers, allow_redirects=True)
                    print(f"PDF response status: {pdf_response.status_code}")  # Debugging response status
                    pdf_response.raise_for_status()  # Raise an error for bad status codes
                    pdf_file_path = os.path.join(download_folder, 'Statement_21_Market_Borrowings.pdf')
                    with open(pdf_file_path, 'wb') as f:
                        f.write(pdf_response.content)
                    print(f"PDF file downloaded successfully to: {pdf_file_path}")
                except requests.exceptions.RequestException as e:
                    print(f"Error downloading PDF file: {e}")
            
            # Break the loop after downloading
            break
else:
    print(f"Failed to retrieve the webpage. Status code: {response.status_code}")


Found the row for Statement 21
Found link: PublicationsView.aspx?id=22286
Found link: https://rbidocs.rbi.org.in/rdocs/Publications/DOCs/21_STATEMENT11122023E9BA62F48A4A43DD88B8024106ECB603.XLSX
Found link: https://rbidocs.rbi.org.in/rdocs/Publications/PDFs/21_STATEMENT11122023E9BA62F48A4A43DD88B8024106ECB603.PDF


## Converting PDF to Desired Format Excel

In [16]:
# Path to the PDF file
pdf_file_path = r"C:/Users/chandresh.kanyal/Downloads/21_STATEMENT11122023E9BA62F48A4A43DD88B8024106ECB603.PDF"

# Output Excel file path
output_excel_path = r"C:/Users/chandresh.kanyal/Desktop/Presentation/Statement- 21.xlsx"

# Parse the PDF for tables
try:
    with pdfplumber.open(pdf_file_path) as pdf:
        tables = []
        for page_num, page in enumerate(pdf.pages, start=1):
            # Extract table from the current page
            table = page.extract_table()
            if table:
                # Convert the table to a DataFrame
                df = pd.DataFrame(table[1:], columns=table[0])  # Use first row as headers
                tables.append((f"Page_{page_num}", df))

        if tables:
            # Save all tables to an Excel file
            with pd.ExcelWriter(output_excel_path) as writer:
                for sheet_name, df in tables:
                    df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"All tables exported successfully to: {output_excel_path}")
        else:
            print("No tables found in the PDF.")
except Exception as e:
    print(f"Error parsing the PDF: {e}")


All tables exported successfully to: C:/Users/chandresh.kanyal/Desktop/excel_converted.xlsx


In [18]:
# Path to the Excel file
file_path = "C:/Users/chandresh.kanyal/Desktop/Presentation/Statement- 21.xlsx"  # Replace with the actual path to your Excel file

# Load the Excel file into a DataFrame
df = pd.read_excel(file_path)

# Print the DataFrame
print(df)


                                            State/UT  \
0                                                NaN   
1                                                  1   
2  1. Andhra Pradesh\n2. Arunachal Pradesh\n3. As...   
3                                 All States and UTs   

                                             2021-22  \
0                               Gross Amount\nRaised   
1                                                  2   
2  46,443.0\n563.0\n12,753.0\n28,489.0\n4,000.0\n...   
3                                         7,01,626.0   

                                          Unnamed: 2  \
0                                         Repayments   
1                                                  3   
2  9,797.8\n33.0\n2,000.0\n4,155.5\n3,087.0\n550....   
3                                         2,09,143.0   

                                             2022-23  \
0                               Gross Amount\nRaised   
1                                            

In [20]:
import pandas as pd

# Load the Excel file into a DataFrame
input_file_path = "C:/Users/chandresh.kanyal/Desktop/Persentation/Presentation- 21.xlsx"  # Replace with the path to your file
df = pd.read_excel(input_file_path)

# Step 1: Clean the DataFrame
# Remove the first two rows (assuming they are metadata)
df_cleaned = df.iloc[2:].reset_index(drop=True)

# Extract state names from the "State/UT" column
state_column = df_cleaned['State/UT'].iloc[0]
states = state_column.split('\n')  # Split by newlines to extract individual state names

# Drop the "State/UT" column (will add it back later)
df_cleaned = df_cleaned.drop(columns=['State/UT'])

# Step 2: Define periods and their corresponding columns
periods_and_columns = [
    ('2021-22', '2021-22', 'Unnamed: 2'),
    ('2022-23', '2022-23', 'Unnamed: 4'),
    ('2023-24*', '2023-24*', 'Unnamed: 6'),
]

# Step 3: Transform data
final_rows = []
for period, gross_col, repayment_col in periods_and_columns:
    # Extract data for "Gross Amount Raised" and "Repayments"
    gross_values = df_cleaned[gross_col].iloc[0].split('\n')
    repayment_values = df_cleaned[repayment_col].iloc[0].split('\n')
    
    # Combine state, period, gross amount, and repayment into rows
    for state, gross, repayment in zip(states, gross_values, repayment_values):
        final_rows.append([state.strip(), period, gross.strip(), repayment.strip()])

# Step 4: Create the transformed DataFrame
transformed_df = pd.DataFrame(final_rows, columns=['State/UT', 'Period', 'Gross Amount Raised', 'Repayments'])

# Step 5: Export to Excel (optional)
output_file_path = "C:/Users/chandresh.kanyal/Desktop/Presentation/Statement- 21_transformed.xlsx"  # Replace with desired output path
transformed_df.to_excel(output_file_path, index=False)
print(f"Transformed data saved to {output_file_path}")

# Display the transformed DataFrame
print(transformed_df)


Transformed data saved to C:/Users/chandresh.kanyal/Desktop/transformed_full_data.xlsx
                State/UT    Period Gross Amount Raised Repayments
0      1. Andhra Pradesh   2021-22            46,443.0    9,797.8
1   2. Arunachal Pradesh   2021-22               563.0       33.0
2               3. Assam   2021-22            12,753.0    2,000.0
3               4. Bihar   2021-22            28,489.0    4,155.5
4        5. Chhattisgarh   2021-22             4,000.0    3,087.0
..                   ...       ...                 ...        ...
83         24. Telangana  2023-24*            27,000.0    5,233.0
84           25. Tripura  2023-24*                   -          -
85     26. Uttar Pradesh  2023-24*            15,500.0    9,434.0
86       27. Uttarakhand  2023-24*             1,000.0    1,200.0
87       28. West Bengal  2023-24*            17,500.0      125.0

[88 rows x 4 columns]


## Converts PDF to Excel (needs manual cleaning before transformation) 
### Transformations can be done using the above code after some manual cleaning

In [22]:
import pdfplumber
import pandas as pd

# Path to the PDF file
pdf_file_path = r"C:/Users/chandresh.kanyal/Downloads/21_STATEMENT11122023E9BA62F48A4A43DD88B8024106ECB603.PDF"

with pdfplumber.open(pdf_file_path) as pdf:
    tables = []
    for page_num, page in enumerate(pdf.pages, start=1):
        # Extract table from the current page
        table = page.extract_table()
        if table:
            # Convert the table to a DataFrame
            df = pd.DataFrame(table[1:], columns=table[0])  # Use first row as headers
            
            # Split cells with newline characters into multiple rows
            new_rows = []
            max_splits = 0
            # First, determine the maximum number of splits any cell has
            for row in df.values:
                for i, cell in enumerate(row):
                    if isinstance(cell, str) and '\n' in cell:
                        num_splits = len(cell.split('\n'))
                        max_splits = max(max_splits, num_splits)
            
            # Now split the cells and align rows accordingly
            for row in df.values:
                # Split each cell by newline if it contains a newline
                split_row = [cell.split('\n') if isinstance(cell, str) and '\n' in cell else [cell] for cell in row]
                
                # Get the number of rows to generate based on the maximum splits
                num_rows = max(len(parts) for parts in split_row)
                
                # Generate the expanded rows based on the splits
                for i in range(num_rows):
                    new_row = [split_row[j][i] if i < len(split_row[j]) else split_row[j][-1] for j in range(len(split_row))]
                    new_rows.append(new_row)
            
            # Recreate the dataframe with the new rows
            df = pd.DataFrame(new_rows, columns=df.columns)
            tables.append((f"Page_{page_num}", df))

# You can now access the tables in the 'tables' list
for page, table_df in tables:

    print(table_df)

output_file_path = "C:/Users/chandresh.kanyal/Desktop/table_df.xlsx"  # Replace with desired output path
table_df.to_excel(output_file_path, index=False)


                 State/UT       2021-22        None       2022-23        None  \
0                    None  Gross Amount  Repayments  Gross Amount  Repayments   
1                    None        Raised  Repayments        Raised  Repayments   
2                       1             2           3             4           5   
3       1. Andhra Pradesh      46,443.0     9,797.8      57,478.0    11,664.0   
4    2. Arunachal Pradesh         563.0        33.0         559.0       170.0   
5                3. Assam      12,753.0     2,000.0      17,100.0       995.0   
6                4. Bihar      28,489.0     4,155.5      36,800.0     9,333.2   
7         5. Chhattisgarh       4,000.0     3,087.0       2,000.0     4,287.0   
8                  6. Goa       2,000.0       550.0       1,350.0       850.0   
9              7. Gujarat      31,054.0    17,500.0      43,000.0    14,700.0   
10             8. Haryana      30,500.0     9,816.7      45,158.0    16,520.0   
11    9. Himachal Pradesh   